# RunPod GPU
Keep **this notebook on your local `.venv` kernel**. Only your training notebook uses the remote Jupyter URL.

## 0 · GPUs and prices (optional, local kernel)
Available GPUs, cheapest first. Copy a name into `GPU` in step 1. Memory needs depend on the model and batch size.

In [1]:
import json, subprocess

gpus = json.loads(subprocess.check_output(["runpodctl", "gpu", "list"]))
for g in sorted(gpus, key=lambda g: g.get("securePricePerHr") or 99):
    if g.get("securePricePerHr") and "NVIDIA" in g["gpuId"] and "MIG" not in g["gpuId"]:
        print(f'${g["securePricePerHr"]}/hr  {g["memoryInGb"]} GB  {g["gpuId"]}')

$0.25/hr  16 GB  NVIDIA RTX A4000
$0.27/hr  24 GB  NVIDIA RTX A5000
$0.28/hr  20 GB  NVIDIA RTX 4000 Ada Generation
$0.5/hr  24 GB  NVIDIA GeForce RTX 3090
$0.57/hr  24 GB  NVIDIA RTX PRO 4000 Blackwell
$0.59/hr  48 GB  NVIDIA A40
$0.59/hr  24 GB  NVIDIA L4
$0.59/hr  48 GB  NVIDIA RTX A6000
$0.72/hr  32 GB  NVIDIA RTX PRO 4500 Blackwell
$0.72/hr  32 GB  NVIDIA RTX PRO 4500 Blackwell Server Edition
$0.89/hr  24 GB  NVIDIA GeForce RTX 4090
$0.96/hr  48 GB  NVIDIA RTX PRO 5000 Blackwell
$0.99/hr  48 GB  NVIDIA RTX 6000 Ada Generation
$1.09/hr  48 GB  NVIDIA L40S
$1.79/hr  80 GB  NVIDIA A100-SXM4-80GB
$2.49/hr  96 GB  NVIDIA RTX PRO 6000 Blackwell Server Edition
$5.29/hr  141 GB  NVIDIA H200
$7.99/hr  180 GB  NVIDIA B200
$8.99/hr  288 GB  NVIDIA B300 SXM6 AC


## 1 · Start a GPU (local kernel)
Copies code, data and saved models; installs pinned packages on Linux. Prints only the ready Jupyter URL. Setup failure deletes this attempt’s pod.
**Charges start now.** Run once; use step 0 for prices.

In [2]:
import json, secrets, shlex, subprocess

GPU = "NVIDIA A40"
IMAGE = "runpod/pytorch:1.0.3-cu1281-torch291-ubuntu2404"  # matches requirements.txt

def run(command):
    result = subprocess.run(command, capture_output=True, text=True, timeout=900)
    if result.returncode:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout

repo = run(["git", "rev-parse", "--show-toplevel"]).strip()
pods = json.loads(run(["runpodctl", "pod", "list", "--all"]))
if any(p["name"].startswith("learn-gpu") for p in pods):
    raise RuntimeError("A learning pod already exists. Use it or delete it first.")

name = "learn-gpu-" + secrets.token_hex(3)
pw = secrets.token_urlsafe(24)
target = "/workspace/LLM-fine-tuning"

try:
    pod = json.loads(run([
        "runpodctl", "pod", "create", "--name", name, "--image", IMAGE,
        "--gpu-id", GPU, "--min-cuda-version", "12.8", "--ports", "8888/http,22/tcp",
        "--container-disk-in-gb", "30", "--volume-in-gb", "50",
        "--env", json.dumps({"JUPYTER_PASSWORD": pw}), "--wait"]))
    ssh = shlex.split(pod["ssh"]["ssh_command"]) + [
        "-o", "StrictHostKeyChecking=accept-new", "-o", "BatchMode=yes",
        "-o", "ConnectTimeout=15", "-o", "ServerAliveInterval=15", "-o", "ServerAliveCountMax=3"]

    # Copy code, data and saved models; skip local environments, secrets and caches.
    skip = ".git .venv __pycache__ .ipynb_checkpoints .DS_Store ._* *.pyc .env .env.* .vscode .idea".split()
    pack = ["tar", "--no-mac-metadata", "--no-xattrs", *["--exclude=" + s for s in skip],
            "-C", repo, "-cf", "-", "."]
    unpack = ssh + [f"mkdir -p {target} && tar --no-same-owner -xf - -C {target}"]
    run(["bash", "-o", "pipefail", "-c", shlex.join(pack) + " | " + shlex.join(unpack)])

    # Install into the pod’s Jupyter Python; confirm CUDA works.
    run(ssh + [f"python -m pip install --break-system-packages --no-cache-dir -r {target}/requirements.txt && "
               "python -c 'import torch; torch.zeros(1).cuda()'"])
    url = f"https://{pod['id']}-8888.proxy.runpod.net/?token={pw}"
    run(["curl", "-fsS", "--retry", "10", "--retry-delay", "2", "--retry-connrefused",
         "--max-time", "10", url.replace("/?", "/api?")])
except BaseException as error:
    # Delete this attempt's pod if setup fails or is interrupted.
    try:
        for p in json.loads(run(["runpodctl", "pod", "list", "--all"])):
            if p["name"] == name:
                run(["runpodctl", "pod", "delete", p["id"]])
        if any(p["name"] == name for p in json.loads(run(["runpodctl", "pod", "list", "--all"]))):
            raise RuntimeError("Pod still exists.")
    except Exception as cleanup_error:
        raise RuntimeError(f"Cleanup failed: delete {name} in the Runpod console. {cleanup_error}") from error
    raise

print(url)


https://y3xrabi4g2qhob-8888.proxy.runpod.net/?token=TUaCqq_woYxc6fN-gG6lOYT7l1uz6Eq4


## 2 · Connect your training notebook
Open `Training/03_instruction.ipynb`: **Select Kernel → Select Another Kernel → Existing Jupyter Server** → paste the URL → choose **Python 3 (ipykernel)**. Keep `setup.ipynb` local.

## Terminal (optional)
In this local kernel, run `print(pod["ssh"]["ssh_command"])`, then paste that command into your terminal. `exit` returns to your Mac.

## 3 · Train
In your training notebook, run its Runpod `%cd` cell, then your model/data/training cells. Its device should print `cuda`.

## Files & scripts
Project: `/workspace/LLM-fine-tuning`. For training notebooks, first run `%cd /workspace/LLM-fine-tuning/Training` (skip locally).

- Cell edits run immediately. Changed local scripts or data need copying again.
- Code/data stay on pod disk; `.to(device)` moves the model and tensor batches to GPU memory.
- To retrieve results: GPU terminal `runpodctl send model.pth` → local terminal `runpodctl receive <code>`. Save results before deleting.

## 4 · Delete the pod (local kernel)
Return here when finished or after a training error. Download results first; this deletes the pod and its disk. If setup failed, skip to step 5.

In [7]:
!runpodctl pod delete {pod['id']}

{"error":"failed to delete pod: pod not found","code":"not_found","status":404}


## 5 · Confirm (local kernel)
`[]` means no pods remain. If a pod is listed, delete its ID with `!runpodctl pod delete <id>`; replace `<id>` with the listed ID. This also works after restarting your local kernel.

In [6]:
!runpodctl pod list --all

[]
